# 数组、元组与只读类型

学习目标：能为列表和位置固定的数据选择数组或元组，并处理越界与只读视图的边界。

前置知识：JavaScript 数组、索引、解构和对象引用；TypeScript 类型标注与联合类型。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。本章附加选项：noUncheckedIndexedAccess=true，含义见对应知识点。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/03-arrays-and-tuples/。

1. [main.ts](scripts/03-arrays-and-tuples/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/03-arrays-and-tuples/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/03-arrays-and-tuples/tsconfig.json)、[tsconfig.errors.json](scripts/03-arrays-and-tuples/tsconfig.errors.json)：分别明确正常与反例文件范围。



Step 1：检查正常项目的类型。

```bash
npm run check:03
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:03
```

Step 3：运行正常示例。

```bash
npm run run:03
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:03
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/03-arrays-and-tuples/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 数组的元素类型与二维结构

数值列表可写成 number[]，也可写成 Array&lt;number&gt;；尖括号中的 number 指定元素类型。数组长度不包含在这种类型里。number[][] 表示每个元素又是数值数组，并不保证各行等长，因此它不是自动校验行列数的矩阵类型。

下面直接遍历行与元素，不依赖索引一定存在。混合列表使用括号包住联合类型：(string | number)[] 的每个元素可为字符串或数值，string | number[] 则允许整个值是字符串或数值数组。

```typescript
export {};
const scores: number[] = [6, 9];
const moreScores: Array<number> = [3];
const rows: number[][] = [scores, moreScores];
const mixed: (string | number)[] = ["课时", 2];
let sum = 0;
for (const row of rows) {
  for (const score of row) sum += score;
}
console.log(sum, mixed.length); // 18 2
```

以下片段来自独立的 type-errors.ts：

```typescript
const badScores: number[] = [1, "2"]; // 元素必须为 number，字符串不会自动转换。
```

## 2 元组记录每个位置的含义

元组（tuple）适合位置有固定含义的短记录。[string, number] 要求首项为字符串、次项为数值；同样的数组字面量若没有元组上下文，通常会推断成联合元素数组。

具名元组元素里的 title、minutes 是工具提示标签，不会生成对象属性，也不要求解构变量同名。二位数值元组中的位置可代表坐标，但顺序仍需由程序约定。本例把第二项定义为非负课时数；number 类型本身并不限制正负。

```typescript
type Lesson = [title: string, minutes: number];
const lesson: Lesson = ["数组", 25];
const [name, duration] = lesson;
const renamed: [topic: string, length: number] = lesson;
console.log(name.toUpperCase(), duration, renamed[1]); // 数组 25 25
```

以下片段来自独立的 type-errors.ts：

```typescript
const fixed: [string, number] = ["数组", 25];
const outside = fixed[2]; // 已知长度为 2，索引 2 不存在。
const reversed: [string, number] = [25, "数组"]; // 两个位置的类型均不匹配。
```

## 3 可选元素、剩余元素与解构

元素名称后的 ? 表示该位置可以省略；读取时要处理 undefined。末尾可选的元组会有有限种可能长度。剩余元素使用数组或元组类型，下面的 ...tags: string[] 表示零个或多个标签，因此该元组没有固定总长度。

普通可选元素不能跟在剩余元素后面，剩余元素后也不能再接第二个剩余元素。剩余元素可以位于开头或中间，只要满足这些限制。TypeScript 5.2 起允许具名与无名元素混用；本章统一加标签，便于阅读。

```typescript
type Reading = [title: string, minutes?: number];
const short: Reading = ["类型"];
const [readingTitle, minutes = 10] = short;
const possibleLength: 1 | 2 = short.length;
type Tagged = [title: string, ...tags: string[]];
const tagged: Tagged = ["元组", "基础", "数据"];
const [taggedTitle, ...tags] = tagged;
const ending: [...labels: string[], count: number] = ["练习", 2];
console.log(readingTitle, minutes, possibleLength, taggedTitle, tags.join("/"), ending.length); // 类型 10 1 元组 基础/数据 2
```

以下片段来自独立的 type-errors.ts：

```typescript
type InvalidTail = [...values: string[], count?: number]; // 剩余元素后不能接可选元素。
```

## 4 索引结果为什么需要检查

本章额外开启 noUncheckedIndexedAccess。对普通数组用数值索引读取时，编译器会把可能缺失的 undefined 加入结果类型；二维数组的行和行内元素都可能不存在。这个选项不负责在运行时检查索引。

已知元组中的必需位置仍保持具体类型；可选位置需要检查。不要用 ! 抹去越界风险。若确实需要随机索引，先保存结果并判断是否为 undefined；若只需要全部元素，使用直接遍历。

```typescript
const first = scores[0]; // number | undefined，即使当前字面量非空。
const absent = rows[4]?.[0]; // 行不存在时，安全返回 undefined。
const checked = first === undefined ? "缺失" : first.toFixed(1);
console.log(checked, absent, lesson[0]); // 6.0 undefined 数组
```

以下片段来自独立的 type-errors.ts：

```typescript
const values: number[] = [];
const unsafe: number = values[0]; // noUncheckedIndexedAccess 使结果包含 undefined。
```

## 5 readonly 是访问视图

“不能通过这个名称 push”与“数组永远不变”是两种不同承诺。沿引用关系看下面的两个名称。

readonly number[] 与 ReadonlyArray&lt;number&gt; 都表示只读数组接口；readonly [string, number] 则保留每个位置的元组信息。可写数组可以交给只读参数，反过来通常不成立，因为接收方可能修改它。

readonly 检查不会冻结数组，不会复制数据，也不会递归禁止元素对象内部的修改。下面可写别名与只读视图指向同一数组；对数组的追加和对元素的修改都能从视图观察到。const 只限制变量重新绑定，与这两类检查也不同。

![readonly 限制访问方式，不复制对象。两种静态访问接口在运行时指向同一个数组。](image/illustration/03-01-readonly-alias.svg)

图示说明：依据 readonly 与别名关系自绘，图展示初始状态；readonly 数组的元素是本例定义的可写对象，不承诺深层只读或冻结。

跟踪 writable.push 和 item.count 的修改，再从 view 读取长度及元素字段，检查静态接口与共享数据的区别。

```typescript
const item = { count: 1 };
const writable = [item];
const view: readonly { count: number }[] = writable;
writable.push({ count: 2 });
item.count += 1; // 两个数组视图中的首元素都引用 item。
const readonlyLesson: readonly [string, number] = lesson;
const names: ReadonlyArray<string> = ["甲", "乙"];
console.log(view.length, view[0]?.count, readonlyLesson[1], names.join(",")); // 2 2 25 甲,乙
```

以下片段来自独立的 type-errors.ts：

```typescript
const readonlyScores: readonly number[] = [1];
readonlyScores.push(2); // 只读数组接口没有 push。
const writableScores: number[] = readonlyScores; // 不能把只读数组交给可写数组类型。
const readonlyPair: readonly [string, number] = ["数组", 1];
readonlyPair[1] = 2; // 只读位置不能通过这个视图赋值。
```

## 本章小结

数组约束元素，元组进一步描述位置和可能的长度。索引读取与遍历的类型信息不同；readonly 限制当前类型视图的写操作，不保证运行时数据不可变。

## 练习

1. 定义依次包含课程名、独立标签数组和末尾可选时长的元组，分别构造省略时长和提供时长的值；检查通过，省略时长时使用默认值 10。
2. 为二维数组实现取值函数，越界返回 undefined；核对空数组、缺失行、缺失列和数值 0 四种输入，不用 !。
3. 将只读数组展开复制成新数组后追加元素；原数组长度应保持不变。再用元素对象验证浅复制仍共享对象。

### 提示

1. 将可选时长放在最后；普通必需元素不能跟在可选元素后面。
2. 两次索引都可能缺失，表达式 rows[row]?.[column] 能保留 undefined。
3. 用 [...view] 得到新数组，再对其中的对象字段做一次修改。


### 参考解析

1. 可用 [title: string, tags: string[], minutes?: number]；例如 ["类型", ["基础"]] 与 ["类型", ["基础"], 25]。解构第三项时给默认值 10。
2. 在本章索引配置下返回类型为 number | undefined；缺行、缺列均返回 undefined，已有数值 0 应原样返回，不能被真值判断误当缺失。
3. 新数组的 push 不增加原数组长度；两边同位置仍引用原来的元素对象，修改对象字段会在两边观察到。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Object Types：Array、The ReadonlyArray Type、Tuple Types](https://www.typescriptlang.org/docs/handbook/2/objects.html#tuple-types)；[noUncheckedIndexedAccess](https://www.typescriptlang.org/tsconfig/noUncheckedIndexedAccess.html)；[4.2：首部与中部剩余元素](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-2.html#leadingmiddle-rest-elements-in-tuple-types)；[5.2：元组标签混用](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-2.html#named-and-anonymous-tuple-elements)。 |
| npm 官方文档 | [npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。 |
